# Oracle-cache enrichment from first principles

Two language models that talk today talk in text. One decodes its internal state into tokens, the other reads those tokens back into a state of its own. Text is the wire format, and it is a lossy one: it throws away most of what the sender actually computed. A natural question follows. Could a model hand another model the state directly, skipping the text? Before building the machinery to do that, one has to establish that the state is worth transferring at all. That is what the oracle measures, and it is what this notebook derives.

The oracle construction: compute the key-value cache for the full sequence $E \oplus X$, an enrichment prompt $E$ followed by the actual input $X$, then discard the $E$ portion and keep only the slice belonging to $X$:

$$
C^{*}(X) \;=\; C\big[\; |E| : |E|+|X| \;\big]\,(E \oplus X)
$$

$X$'s keys and values were computed while attending to $E$, so the retained slice carries a trace of $E$ even after $E$'s own entries are gone. It is an upper bound rather than a method: it uses the true in-context cache, not a learned projection, so it answers the narrower question of whether a perfectly enriched cache helps at all, before anyone builds the harder thing.

This notebook derives why that trace exists, where it lives in the tensors, how the position metadata is silently lost when the slice is moved, and only at the end whether the trace is large enough to change an answer.

One rule holds throughout. No accuracy number appears before the final section. Every claim is paired with code that runs, and where the notebook shows a function it reads that function from the canonical module with `inspect.getsource` rather than retyping it, so the page cannot drift from the code.

<!-- Working note, not for the reader. Internal ordering behind the sections:
     intuition, derivation, math-to-tensor bridge, bug autopsy, measurement.
     The reader should only see the content headings, never these labels. -->

### What a cache is, from the ground up

The whole notebook rests on three ideas, so they are worth stating plainly first.

**Attention.** For each token the model builds three vectors: a *query* $q$, a *key* $k$, and a *value* $v$. To decide how much a token should draw from an earlier token, it compares the later token's query with the earlier token's key by a dot product. Large dot product means high relevance. Those comparisons are turned into weights that sum to one, and the token's new content is the weighted sum of the earlier *values*. Keys advertise what a token offers; queries ask for it; values are what actually gets mixed in.

**The KV cache.** When the model generates text one token at a time, every new token compares itself against the keys and values of everything before it. Those earlier keys and values do not change as generation proceeds, so recomputing them each step would be waste. They are computed once and kept. That store is the key-value cache, and it has a fixed shape: one key vector and one value vector per token, per layer, per attention head.

**Prefill and decode.** Reading the prompt and filling the cache in one parallel pass is *prefill*. Generating new tokens one at a time, each reading the cache and appending its own entry, is *decode*. The cache is therefore not a transcript of the prompt. It is the model's reading of the prompt: shaped by position, by layer, and by everything each token was allowed to attend to while it was computed. That last clause is the hinge the entire oracle turns on.

In [ ]:
import os
import subprocess
import sys
import inspect
import base64

import numpy as np
from IPython.display import Code, Image, HTML

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "."))
ASSETS = os.path.join(REPO_ROOT, "assets")

# Canonical modules live at repo root. Adjust if they move to src/.
sys.path.insert(0, REPO_ROOT)


def show_source(obj):
    '''Render a function's source straight from its module, syntax highlighted.'''
    return Code(inspect.getsource(obj), language="python")


def run_test(name):
    '''Run a test script as a subprocess and return its exit code.

    The test prints nothing on success and exits 0. Any output here is a
    failure report. REPO_ROOT is placed on PYTHONPATH so the subprocess can
    import the project's modules: a child process does not inherit the
    sys.path edit above, and running a script by path puts only the script's
    own directory on the path, not the repo root.
    '''
    env = dict(os.environ)
    env["PYTHONPATH"] = REPO_ROOT + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run(
        [sys.executable, os.path.join(REPO_ROOT, name)],
        capture_output=True, text=True, cwd=REPO_ROOT, env=env,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
    return result.returncode


def show_gif(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    return HTML(f'<img src="data:image/gif;base64,{b64}" />')


def show_image(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    return HTML(f'<img src="data:image/png;base64,{b64}" />')

These four helpers are the notebook's whole toolkit. `show_source` reads a function's text directly from the imported module, so what you see on the page is what the code actually is; there is no second copy to fall out of date. `run_test` runs a test file the way a shell would and keeps its exit code, following the convention that a passing test is silent and returns 0. It also puts the repo root on the subprocess's `PYTHONPATH`, so a test launched by path can still import the project's modules (a subprocess does not inherit the `sys.path` line above). `show_gif` and `show_image` embed each figure as base64 inside the notebook itself, so the finished document is self-contained and needs no sidecar files to display.

---

## 1. A cache slice carries more than its tokens

Few-shot prompting is the observation that placing a few worked examples $E$ before a question $X$ tends to raise accuracy. The usual account is attention: while answering, the model can look back at the examples. But a second mechanism hides inside the first. When the model prefills the concatenation $E \oplus X$, the cache entries for $X$ are computed *in the presence of* $E$. The question tokens attend to the example tokens during that single prefill pass, so $X$'s own keys and values already carry a trace of the examples before a single answer token is produced.

That splits the benefit of few-shot prompting into two separable parts:

$$
\underbrace{\text{H1}}_{\text{attend to } E \text{ during decode}}
\qquad
\underbrace{\text{H2}}_{\text{enrichment of } X\text{'s cache during prefill}}
$$

H1 requires $E$ to remain in the cache, because you cannot attend to what is not there. H2 survives even if $E$ is deleted afterward, because it lives in $X$'s own entries. If H2 is real on its own, then a model can be helped without lengthening its cache, and the useful information is in the cache entries themselves rather than in the tokens they point back to. That is precisely the property that would make representation-level transfer between models conceivable.

The three conditions isolate these parts. All run on the same model and the same questions, and differ only in the cache they decode from:

$$
|C_{\text{Direct}}| = |X|, \qquad
|C_{\text{Few-shot}}| = |E| + |X|, \qquad
|C_{\text{Oracle}}| = |X|
$$

Direct reads the question cold. Few-shot keeps the whole enriched cache, so both H1 and H2 are active. Oracle prefills $E \oplus X$ exactly as Few-shot does, then slices $E$ out before decoding: same length as Direct, but every entry was computed while $E$ was visible. Oracle removes H1 and keeps H2. Comparing Oracle against Direct is therefore a clean test of H2 alone.

In [ ]:
def cache_length(len_E, len_X, condition):
    # The cache the model decodes from, measured in tokens.
    if condition == "direct":
        return len_X               # question only
    if condition == "few_shot":
        return len_E + len_X       # examples plus question
    if condition == "oracle":
        return len_X               # sliced back down to the question's length
    raise ValueError(condition)

for cond in ("direct", "few_shot", "oracle"):
    print(f"{cond:9s} {cache_length(len_E=180, len_X=60, condition=cond):>4d} tokens")

With a 180-token example block and a 60-token question, Few-shot carries 240 cache tokens while Direct and Oracle each carry 60. Oracle and Direct are indistinguishable on this axis by construction. The entire cost gap between them and Few-shot is the example block $E$, which is exactly the cost Oracle refuses to pay while trying to keep the benefit.

In [ ]:
show_image(os.path.join(ASSETS, "cache_length_distribution.png"))

Across a real run, Direct and Oracle sit on the same short distribution while Few-shot spreads much longer and much wider. The figure is on a log scale, so the separation is larger than it looks. This is the ledger the rest of the notebook prices: Oracle promises Few-shot's enrichment at Direct's cache length, and the question is whether that promise holds.

---

## 2. A trace of the prompt persists in $K_X$

Take one token of $X$ at local position $t$, in layer $\ell$. Its cached key is a fixed linear map of its hidden state:

$$
K^{\ell}_t = W^{\ell}_K \, h^{\ell}_t
$$

The *hidden state* $h^{\ell}_t$ is the token's running representation, the vector carried up the network that every layer reads from and adds to. Its update at each layer is a residual sum: the previous value, plus the attention output, plus a small feed-forward term.

$$
h^{\ell+1}_t = h^{\ell}_t + o^{\ell}_t + \mathrm{MLP}(\cdot)
$$

In the full context $E \oplus X$, the token sits at absolute position $m + t$ with $m = |E|$, and its attention output sums over every earlier position, the examples included:

$$
o^{\ell}_t \;=\; \sum_{j \le m+t} \alpha_{tj}\, W^{\ell}_V h^{\ell}_j
\;=\; \underbrace{\sum_{j=1}^{m} \alpha_{tj}\, W^{\ell}_V h^{\ell}_j}_{\text{contribution from } E}
\;+\; \sum_{j=m+1}^{m+t} \alpha_{tj}\, W^{\ell}_V h^{\ell}_j
$$

Here $\alpha_{tj}$ are the attention weights, the shares that sum to one, and the first sum runs over the example positions. As long as the token places any weight on the examples, that first sum is nonzero, so $h_t$ differs from what it would have been alone, and since the key is a fixed map of the hidden state, the key differs too:

$$
\Delta K^{\ell}_t \;=\; K^{\ell}_t(E \oplus X) - K^{\ell}_t(X) \;\ne\; 0
$$

The residual sum then carries that difference upward, so it compounds across layers rather than appearing at one.

This is the mechanism, and it is also honest about its own limit. The argument shows $\Delta K \ne 0$. It does **not** show that $\Delta K$ is large in the directions a later query actually reads. A difference can exist and still be too small to change any answer. That magnitude question is empirical, and it is what the sweeps in this section, and the measurement in the last one, are for.

In [ ]:
# A minimal illustration that changing a hidden state changes its key, and by
# how much. W_K is a fixed linear map; h is the hidden state; nudging h by the
# kind of contribution attention to E would add rotates the resulting key.
rng = np.random.default_rng(0)
d = 16
W_K = rng.standard_normal((d, d))

h_alone = rng.standard_normal(d)          # X token prefilled by itself
e_effect = 0.6 * rng.standard_normal(d)   # what attending to E adds to h
h_in_context = h_alone + e_effect         # same token, now computed after E

K_alone = W_K @ h_alone
K_context = W_K @ h_in_context

def cosine_similarity(a, b):
    # cosine of the angle between two vectors: +1 same direction, 0 orthogonal,
    # -1 opposite. It reads direction only and ignores length.
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("cosine similarity K_alone vs K_context:", round(cosine_similarity(K_alone, K_context), 3))

The two keys belong to the same token; the only difference is that one was computed after $E$ and one was not. Their cosine similarity is well below 1, meaning they point in noticeably different directions. *Cosine similarity* is the cosine of the angle between two vectors: it answers "did this point somewhere else" rather than "did this get bigger", which is the right question here, since attention compares directions through a dot product. This is $\Delta K \ne 0$ made concrete on a toy vector. The figures below ask the same question of the real model.

In [ ]:
show_gif(os.path.join(ASSETS, "h_x_contamination.gif"))

Cosine distance (which is $1 - \text{cosine similarity}$, so larger means more divergence) between $h_X$ computed alone and $h_X$ computed after $E$, layer by layer. At the embedding output the two are identical, because no attention has run yet and the example sum above is empty. The distance then grows as layers accumulate attention over $E$. The trace is not injected at one place; it builds up the stack, exactly as the residual sum predicts.

In [ ]:
show_image(os.path.join(ASSETS, "k_x_context_comparison.png"))

The same token, position 0 of $X$, at one layer and one key-value head, with its key drawn twice: computed alone, and computed after $E$. The two point in substantially different directions, cosine similarity about $-0.847$ here. This is a single head, layer, and token, chosen because the effect is vivid on it, not because one head settles the magnitude question. The next figure aggregates.

In [ ]:
show_gif(os.path.join(ASSETS, "k_x_cosine_distance_sweep.gif"))

The same comparison averaged across all 8 key-value heads and swept across layers. This is the aggregate version of the previous panel, and it shows the divergence is present through the stack rather than an artifact of the one head picked above. It remains a statement about geometry. Whether that geometry moves an answer is still deferred to the final section.

---

## 3. Where the trace lives in the tensors

The symbols above are rows and axes of real tensors. For Qwen3-0.6B the cache has 28 layers, 8 key-value heads, and a head dimension of 128, and every cache tensor has shape $[\,b,\; n_{kv},\; \text{seq},\; d\,]$.

Two architectural facts are worth naming from first principles, because they matter the moment a second model enters the picture. *Grouped-query attention* is why there are 16 query heads but only 8 key-value heads: several query heads share one key-value head, which shrinks the cache without shrinking the query-side computation. *Head dimension* is the width of each head's vectors; in Qwen3 it is set as an explicit config field ($d = 128$) and is not the hidden size divided by the head count (that ratio would give 64). Reading it off the config rather than deriving it is the difference between a correct slice and a silently wrong one.

| Symbol | Meaning | Tensor location |
|---|---|---|
| $E$ | enrichment prompt, $m = |E|$ tokens | `input_ids[:, :m]` |
| $X$ | input, $n = |X|$ tokens | `input_ids[:, m:m+n]` |
| $C(E \oplus X)$ | full prefill cache | `cache.layers[l].keys` / `.values`, sequence axis length $m+n$ |
| $K_X$ | key rows belonging to $X$ | `layer.keys[:, :, m:m+n, :]` |
| $C^{*}(X)$ | sliced cache | keep `[:, :, m:m+n, :]` for keys and values, every layer |
| first decode position | $m+n$, not $n$ | `position_ids = m + n` (next section) |

The slice is one array operation per layer, applied identically to keys and values. Here is the canonical function that does it, read straight from the module:

In [ ]:
from cache_enrichment import slice_cache
show_source(slice_cache)

`slice_cache` keeps the sequence-axis window `[len_E : len_E + len_X]` for both the key and the value tensor at every layer, and touches nothing else. The head and head-dimension axes are untouched, so the sliced cache is structurally identical to a fresh Direct cache of the same length; only its contents differ. The next cell shows that the slice really is nothing more than indexing on one axis.

In [ ]:
# The slice, on a dummy cache-shaped array, to make the axes concrete.
b, n_kv, d = 1, 8, 128
len_E, len_X = 6, 5
seq = len_E + len_X

keys_full = np.zeros((b, n_kv, seq, d))          # C(E oplus X), one layer
keys_sliced = keys_full[:, :, len_E:len_E + len_X, :]   # C*(X): keep X's rows

print("full   ", keys_full.shape)
print("sliced ", keys_sliced.shape)
print("dropped rows on the sequence axis:", seq - keys_sliced.shape[2])

Only the sequence axis changes, from $m+n$ down to $n$. Batch, head, and head-dimension axes are identical before and after. The slice discards $E$'s $m$ rows and keeps $X$'s $n$, which is the whole operation. A shape test pins this for both models' head configurations so a wrong axis cannot pass silently:

In [ ]:
run_test("tests/test_slice_shape.py")

No output and a return value of 0 mean the test passed. The convention is deliberate: a passing test says nothing, so anything printed here would be a failure to read. The test checks the sliced shape at every layer for both Qwen2.5-0.5B's and Qwen3-0.6B's head configurations, because the same slicing code is meant to be reused unchanged once a second model joins.

In [ ]:
show_image(os.path.join(ASSETS, "tensor_slicing_schematic.png"))

The retained slice keeps the original absolute positions $|E| \dots |E|+|X|-1$. The failure mode drawn at the bottom, relabelling them $0 \dots |X|-1$, loses the metadata that says where these tokens really were. That relabelling is the bug the next section dissects.

---

## 4. Moving a slice without its position corrupts it silently

**Rotary position embedding, briefly and from scratch.** A dot product between a query and a key reads their contents but has no notion of where either token sits, so position has to be injected. Rotary position embedding (RoPE) injects it by *rotating* each query and key by an angle proportional to its absolute position. Because the dot product of two rotated vectors depends only on the difference of their angles, the score between a query at position $p$ and a key at position $j$ ends up depending only on the relative distance $p - j$. Position goes in as an absolute rotation; relative distance comes out for free.

Now the bug. RoPE is applied to each key at prefill, so the cached key already has its absolute-position rotation baked in:

$$
\tilde{k}_j = R_{p_j}\, k_j, \qquad p_j = m + j \ \text{ for retained } X\text{-token } j
$$

A decode-step query at position $p_q$ scores against it through the difference of rotations:

$$
\text{score} \;\propto\; (R_{p_q} q)^{\top} (R_{p_j} k_j) \;=\; q^{\top} R_{\,p_j - p_q}\, k_j
$$

The first decoded token belongs at $p_q = m + n$, giving a true relative distance $d_{\text{correct}} = p_q - p_j = (m+n) - (m+j) = n - j$. Reset that query to $p_q = n$, as if the sliced cache started at zero, while the cached keys keep their baked-in $m + j$, and every distance shifts by the same constant:

$$
d_{\text{bug}} = n - (m+j) = (n - j) - m = d_{\text{correct}} - |E|
$$

Because a rotation is orthogonal ($R^{\top} R = I$, so $\lVert R v \rVert = \lVert v \rVert$), the wrong offset produces no error, no NaN, and no change in vector length. Nothing surfaces except quietly wrong scores. This is why the code separates the two indices by hand instead of calling `model.generate()`, which would derive one from the other and collapse exactly the distinction under test.

**Evaluation-mode collapse.** There is a second subtlety in how this gets measured. The first decoded token is read from the logits of the last prefill position, computed once on the unsliced $E \oplus X$, before the slice is applied. Few-shot and Oracle share that tensor, so

$$
\text{token}_1(\text{Few-shot}) = \text{token}_1(\text{Oracle}) \quad \text{exactly, for every sample}
$$

Since the scored multiple-choice answer is that very first token, this harness cannot separate Few-shot from Oracle: it resolves $\text{Direct}$ against $\{\text{Few-shot}, \text{Oracle}\}$. The two only diverge from the second token onward, where the sliced cache is first read. The canonical decode loop makes both points visible in code:

In [ ]:
from run_experiment import decode_loop
show_source(decode_loop)

Two things to read here. First, `position` is built from `start_position + step` while `cache_position` is built from `cache.get_seq_length()`. These are two different numbers for Oracle: the cache holds $|X|$ entries, but the position id starts at $|E| + |X|$. Keeping them as separate arguments is the entire fix for the position bug. Second, the first `next_token` is taken from `first_logits` before the loop runs, which is why Few-shot and Oracle, handed the same prefill logits, must produce the same first token. The eval-mode collapse is not an accident of the data; it is written into the control flow.

In [ ]:
# RoPE in two dimensions: a rotation matrix, and what a reset position does to a
# score. Numbers match the animation below (offset = len_E = 6).
def R(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

theta = 0.5
len_E, len_X, j = 6, 5, 0
key_pos = len_E + j            # absolute position baked into the cached key
q_correct = len_E + len_X      # 11: where the first decoded token really is
q_bug = len_X                  # 5: the reset that ignores the discarded E

q = np.array([1.0, 0.0])
k = np.array([1.0, 0.0])
k_cached = R(key_pos * theta) @ k          # key was rotated at prefill

def score(query_pos):
    return float((R(query_pos * theta) @ q) @ k_cached)

print(f"relative distance  correct = {q_correct - key_pos:+d}   bug = {q_bug - key_pos:+d}")
print(f"score              correct = {score(q_correct):+.4f}   bug = {score(q_bug):+.4f}")
print(f"key length         before  = {np.linalg.norm(k):.4f}   after rotation = {np.linalg.norm(k_cached):.4f}")

The correct query sees the retained key at relative distance $+5$; the reset query sees it at $-1$, off by exactly $|E| = 6$. The scores differ accordingly. The last line is the reason the bug is so dangerous: the key's length is unchanged by rotation, so the wrong offset leaves no NaN and no exception behind, only a wrong number in a place nothing checks. A regression test pins the invariant that the position and the cache length differ by exactly $|E|$:

In [ ]:
run_test("tests/test_position_correction.py")

Again, silence and a 0 exit code mean the invariant held on every sample the test constructs. The check is `oracle_pos - oracle_cache == len_E`, which is the arithmetic form of "the cache got shorter but the position did not move".

In [ ]:
show_gif(os.path.join(ASSETS, "rope_position_bug.gif"))

The same query and retained key drawn with the correct position and with the reset one. The offset is the constant $|E|$: the animation uses $|E| = 6$, turning a true relative distance of $+5$ into $-1$, matching the code cell above. Both vectors keep their length in both panels, which is why the error is invisible anywhere except in the score itself.

In [ ]:
show_image(os.path.join(ASSETS, "attention_heatmap_comparison.png"))

A real Qwen3-0.6B capture at layer 27. With correct positions the decode query concentrates its attention on the most recent retained keys, the bright column, with a peak weight of $0.183$. With the reset bug that concentration collapses to a peak of $0.027$, and the query can no longer localize within its own retained context. The difference panel on the right isolates where the attention mass moved. These are attention weights, not accuracy; the accuracy consequence is left for the final section.

In [ ]:
show_gif(os.path.join(ASSETS, "eval_mode_collapse.gif"))

Step one of decode: the first-token logits are identical for Few-shot and Oracle, both topped by `' D'` against a gold answer of `D`. This is the collapse from the decode loop, made visible. Because the equality is guaranteed by the shared prefill rather than merely observed, the downstream extractor does not have to anticipate every degenerate continuation; for these two conditions it only needs the first character, which is already fixed.

---

## 5. Does the trace change an answer?

This is the only section that may state an accuracy number. Section 4 showed that this harness cannot separate Few-shot from Oracle, so the comparison that carries meaning is $\text{Direct}$ against $\text{Oracle}$: same cache length, differing only in whether the entries were computed after $E$. Because both conditions answer the same questions, the comparison is *paired*, meaning each question is scored under both conditions and the difference is read within a question rather than across two separate pools.

Score each sample under each condition as correct or not, $c_i \in \{0, 1\}$. Sort the paired results into four cells. The two cells where both conditions agree carry no information about which is better; only the two *discordant* cells, where exactly one condition is right, do. The paired accuracy difference is built entirely from them:

$$
\hat{\delta} = \frac{1}{N} \sum_{i=1}^{N} \big( c^{\text{Oracle}}_i - c^{\text{Direct}}_i \big) = \frac{n_{01} - n_{10}}{N}
$$

following the code's naming: $n_{01}$ is Direct wrong and Oracle right, $n_{10}$ is Direct right and Oracle wrong. Two questions then follow. Is that split lopsided enough to be real, and how wide is the uncertainty on $\hat{\delta}$?

*McNemar's test* answers the first, and it is a coin-flip argument at heart. If the two conditions were truly equal, each discordant sample would fall to one side or the other like a fair coin, so the number landing on Oracle's side would follow a $\text{Binomial}(n_{01} + n_{10},\ 0.5)$. The p-value is the probability of a split at least this lopsided from fair coins. The canonical code uses the *exact* binomial rather than the chi-square approximation, because the discordant count is small enough that the approximation is not trustworthy and the exact test costs nothing at this scale.

*The bootstrap* answers the second. Rather than assume a formula for the spread of $\hat{\delta}$, it resamples the paired samples with replacement many times, recomputes $\hat{\delta}$ on each resample, and reads the middle 95% of those values as the interval. Resampling *samples* and reading both conditions at each drawn index is what preserves the pairing; resampling the two conditions independently would discard it and widen the interval for the wrong reason.

The three canonical functions, read from the module:

In [ ]:
from analysis import mcnemar, bootstrap_ci, classify_outcome
show_source(mcnemar)

`mcnemar` counts the two discordant cells, drops the concordant ones, and runs the exact two-sided binomial test on the split. `n01` is `(~a & b)`, Direct wrong and Oracle right; `n10` is `(a & ~b)`, Direct right and Oracle wrong. If nothing is discordant it returns a p-value of 1, since there is then no evidence of any difference at all.

In [ ]:
# The statistics on a small paired example, to show the mechanics end to end.
# a = Direct correct per sample, b = Oracle correct per sample (same questions).
a = np.array([1, 1, 0, 0, 1, 0, 0, 1, 0, 1], dtype=bool)  # Direct
b = np.array([1, 0, 1, 1, 1, 1, 0, 1, 1, 1], dtype=bool)  # Oracle

delta = b.mean() - a.mean()
lo, hi, _ = bootstrap_ci(a, b, seed=0)
mc = mcnemar(a, b)

print(f"delta            {delta:+.3f}")
print(f"bootstrap 95% CI [{lo:+.3f}, {hi:+.3f}]")
print(f"mcnemar n01/n10  {mc['n01']}/{mc['n10']}   p = {mc['p_value']:.4f}")
print(f"outcome          {classify_outcome(lo, hi)}")

On this toy set Oracle wins more of the disagreements than Direct, so $\hat{\delta}$ is positive. `classify_outcome` reads only the interval, never the point estimate: outcome **A** when the whole interval sits above zero (Oracle established as better), **B** when it sits below (Direct better), and **C** when it spans zero, meaning the sign is simply not resolved at this sample size. An interval that includes zero is not a null result dressed up; it is an honest statement that the data cannot yet tell the direction. With ten samples this toy interval is wide, which is the point the real run has to overcome with scale.

In [ ]:
show_image(os.path.join(ASSETS, "mcnemar_table.png"))

On the real run at $N = 2000$, the concordant cells carry no signal and the two discordant cells do. Oracle wins the disagreements $345$ to $260$, and that margin is the whole of $\hat{\delta}$. (This asset is drawn from the pre-extractor-fix run; the numbers cited in the closing paragraph are post-fix, and the figure is due to be regenerated from the post-fix data.)

In [ ]:
show_image(os.path.join(ASSETS, "bootstrap_histogram.png"))

The resampled distribution of $\hat{\delta}$ sits entirely above zero, which is the interval form of the same finding and the reason the run classifies as outcome A. At an earlier $N = 100$ the same interval spanned zero, so the sign was unresolved; it was scale, not the extractor fix, that moved it across. (Pre-fix asset, to be regenerated.)

**The number, and its boundaries.** On the post-extractor-fix run at $N = 2000$: Direct $0.454$, Oracle $0.498$, paired difference $\hat{\delta} = +0.044$, bootstrap 95% CI $[+0.021, +0.068]$, McNemar $p = 0.0003$. The interval excludes zero, so the outcome is **A**: at matched cache length, Oracle is measurably more accurate than Direct. H2 is real on its own.

What this does not say matters as much as what it does. It is a claim about accuracy at matched cache length, for one model reading back a slice of its own cache. It is not a claim about decode speed, not a claim about a trained projection or fuser (there is none anywhere in this pipeline), and above all not a claim about transferring a cache between two *different* models, which this same-model oracle was never built to test. The oracle establishes that the asset being transferred exists and has measurable value. The cost of converting it into another model's representation is the next question, not this one.

---

## Closing gates

1. No accuracy number appears before this final section.
2. Every function shown is read from its canonical module with `inspect.getsource`, not retyped, so the page cannot drift from the code it describes.